# Library Imports

In [1]:
import pandas as pd
import glob
import requests
import time

In [2]:
import seaborn as sns

# How to get the data? and which data I'm using?

<div style="float: left">
<img style="float: right" src="datasets/privacy_page.jpg" alt="screenshot" width="600"/>
Good question! I actually just found this out recently myself that you can request your streaming data from Spotify! 

The process is quite simple, All you need to do is to access [your account's privacy page](https://www.spotify.com/us/account/privacy/) , put a tick on the data that you want to request from spotify (see the screenshot on the right)

The data I'm using in this project is the streaming hisotry (from Extended Streaming History option) where it consists of multiple .json files of your spotify streaming history (for both music and podcasts)
I chose the Extended Streaming History because the data includes your entire lifetime of your account, and it also includes _spotify_track_uri_, a unique identifier for each spotify tracks that we can use to collect even more information about the tracks (such as audio features, track info, etc) using [Spotify Web API](https://developer.spotify.com/) later.

Account Data only conists of one-year length of data and it does not have the _spotify_track_uri_ information, technically you can still collect the tracks' info, audio features data using only artists' name and tracks' name but it adds more work and can be quite unreliable (plus, extended streaming history gives you MORE data, so why not?)

Based on my experience, it took about 10 days to receive the Account Data and about 3 weeks to get Extended Streaming History from the time I requested the data.
</div>


# Data Imports / Exports

In [5]:
# THESE FILES ARE EXCLUDED FROM THE REPOSITORY
stream_json = glob.glob('datasets/v3/Streaming_History_Audio*.json')

In [6]:
# see the list of globbed files
stream_json

['datasets/v3/Streaming_History_Audio_2018-2019_2.json',
 'datasets/v3/Streaming_History_Audio_2020-2021_4.json',
 'datasets/v3/Streaming_History_Audio_2019-2020_3.json',
 'datasets/v3/Streaming_History_Audio_2017-2018_1.json',
 'datasets/v3/Streaming_History_Audio_2016-2017_0.json',
 'datasets/v3/Streaming_History_Audio_2024-2025_9.json',
 'datasets/v3/Streaming_History_Audio_2022-2023_7.json',
 'datasets/v3/Streaming_History_Audio_2022_6.json',
 'datasets/v3/Streaming_History_Audio_2023-2024_8.json',
 'datasets/v3/Streaming_History_Audio_2021-2022_5.json']

In [7]:
# load the globbed files into a dataframe
stream_df = pd.concat([pd.read_json(f) for f in stream_json])

In [ ]:
# these files are for testing purpose (running this cell is not necessary)
tracks_df = pd.read_csv('datasets/tracks_df.csv')
podcasts_df = pd.read_csv('datasets/podcasts_df.csv')

C:\Users\nazhi\AppData\Local\Temp\ipykernel_5344\2056156176.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  tracks_df = pd.read_csv('datasets/tracks_df.csv')


# Preparing for Data Collection
we need 4 things here.
1. tracks streaming history
2. podcasts streaming history
3. list of unique tracks streamed
4. ~~list of unique podcasts streamed~~ not really needed actually

In [8]:
print(stream_df.shape)
print(stream_df.columns)

(153913, 23)
Index(['ts', 'platform', 'ms_played', 'conn_country', 'ip_addr',
       'master_metadata_track_name', 'master_metadata_album_artist_name',
       'master_metadata_album_album_name', 'spotify_track_uri', 'episode_name',
       'episode_show_name', 'spotify_episode_uri', 'audiobook_title',
       'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',
       'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline',
       'offline_timestamp', 'incognito_mode'],
      dtype='object')


In [9]:
stream_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 153913 entries, 0 to 15303
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   ts                                 153913 non-null  object 
 1   platform                           153913 non-null  object 
 2   ms_played                          153913 non-null  int64  
 3   conn_country                       153913 non-null  object 
 4   ip_addr                            153913 non-null  object 
 5   master_metadata_track_name         153246 non-null  object 
 6   master_metadata_album_artist_name  153246 non-null  object 
 7   master_metadata_album_album_name   153246 non-null  object 
 8   spotify_track_uri                  153246 non-null  object 
 9   episode_name                       667 non-null     object 
 10  episode_show_name                  667 non-null     object 
 11  spotify_episode_uri                667 non-nu

In [11]:
# number of unique values in each column
stream_df.nunique()

ts                                   146332
platform                                 34
ms_played                             29401
conn_country                              9
ip_addr                                3842
master_metadata_track_name             5193
master_metadata_album_artist_name      1745
master_metadata_album_album_name       2953
spotify_track_uri                      5723
episode_name                            313
episode_show_name                        37
spotify_episode_uri                     313
audiobook_title                           0
audiobook_uri                             0
audiobook_chapter_uri                     0
audiobook_chapter_title                   0
reason_start                             10
reason_end                               10
shuffle                                   2
skipped                                   2
offline                                   2
offline_timestamp                     51962
incognito_mode                  

In [12]:
stream_df.isna().sum()

ts                                        0
platform                                  0
ms_played                                 0
conn_country                              0
ip_addr                                   0
master_metadata_track_name              667
master_metadata_album_artist_name       667
master_metadata_album_album_name        667
spotify_track_uri                       667
episode_name                         153246
episode_show_name                    153246
spotify_episode_uri                  153246
audiobook_title                      153913
audiobook_uri                        153913
audiobook_chapter_uri                153913
audiobook_chapter_title              153913
reason_start                              0
reason_end                                0
shuffle                                   0
skipped                                   0
offline                                   0
offline_timestamp                    101699
incognito_mode                  

In [17]:
# separate the tracks, podcasts and audiobooks
tracks_df = stream_df.loc[stream_df["spotify_track_uri"].notna(), :]
podcasts_df = stream_df.loc[stream_df["spotify_episode_uri"].notna(), :]
audiobooks_df = stream_df.loc[stream_df["audiobook_uri"].notna(), :]

In [ ]:
print(tracks_df.shape)
print(podcasts_df.shape)
print(audiobooks_df.shape)

(153246, 23)
(667, 23)
(0, 23)


In [19]:
# list of tracks, containing unique values in 'spotify_track_uri' column
tracks_uri_list = tracks_df['spotify_track_uri'].unique().tolist()
podcasts_uri_list = podcasts_df['spotify_episode_uri'].unique().tolist()
audiobooks_uri_list = audiobooks_df['audiobook_uri'].unique().tolist()

In [20]:
print(len(tracks_uri_list))
print(len(podcasts_uri_list))
print(len(audiobooks_uri_list))

5723
313
0


In [24]:
# save tracks_df and podcasts_df into csv
tracks_df.to_csv('datasets/tracks_df.csv', index=False)
podcasts_df.to_csv('datasets/podcasts_df.csv', index=False)
# audiobooks_df.to_csv('datasets/audiobooks_df.csv', index=False)

# Collecting tracks' info using Spotipy
now we're going to collect the using [Spotipy](https://spotipy.readthedocs.io/en/2.22.1/), it'll require user credentials (client_id and client_secret) which you can get [here](https://developer.spotify.com/dashboard) by registering your app at the dashboard

## Setting up Spotipy

In [60]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [131]:
from config import SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET
client_id = SPOTIFY_CLIENT_ID
client_secret = SPOTIFY_CLIENT_SECRET

In [133]:
#Authentication - without user
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager = client_credentials_manager)

## Prepare track_uri_list

here we are going to prepare the list of unique track_uri from the tracks_df, we'll check if we have previously collected the data or not, if we have, we'll just collect the tracks' info that we haven't collected yet

In [119]:
# load tracks_df.csv and get the uris
tracks_df = pd.read_csv('datasets/tracks_df.csv')

# get the uris from it
tracks_uri_list = tracks_df['spotify_track_uri'].unique().tolist()

In [120]:
# check if we have already collected tracks' info data before
try:
    tracks_info_df = pd.read_csv('datasets/tracks_info_df.csv')
except FileNotFoundError:
    tracks_info_df = None

In [121]:
# get the uris that are not in the tracks_info_df
if tracks_info_df is not None:
    tracks_info_uri_list = tracks_info_df['track_id'].unique().tolist()
    # add 'spotify:track:' prefix to the uris
    tracks_info_uri_list = ['spotify:track:' + track_id for track_id in tracks_info_uri_list]
    # get the uris that are not in the tracks_info_df
    tracks_uri_list = list(set(tracks_uri_list) - set(tracks_info_uri_list))

## Collecting Track Info

In [69]:
# since they have the batch request limit of 50, we need to split the list into batches of 50
track_info = {}
for i in range(0, len(tracks_uri_list), 50):
    track_batch = tracks_uri_list[i:i+50]
    sapi = sp.tracks(track_batch)
    for track in sapi['tracks']:
        track_info[track['id']] = {
            'name':track['name'], 
            'artistName':track['artists'][0]['name'], 
            'release_date':track['album']['release_date'], 
            'popularity':track['popularity'], 
            'duration_ms':track['duration_ms'], 
            'isrc':track['external_ids'].get('isrc', None)
            }
    time.sleep(1)
    

In [70]:
len(track_info)

478

In [85]:
list(track_info.items())[:2]

[('0ftCbviWA5aWopBjsBK6Pv',
  {'name': "I Don't Get to Say I Love You Anymore",
   'artistName': 'Tessa Violet',
   'release_date': '2016-10-14',
   'popularity': 25,
   'duration_ms': 210013,
   'isrc': 'ushm21662312'}),
 ('6dOtVTDdiauQNBQEDOtlAB',
  {'name': 'BIRDS OF A FEATHER',
   'artistName': 'Billie Eilish',
   'release_date': '2024-05-17',
   'popularity': 99,
   'duration_ms': 210373,
   'isrc': 'USUM72401994'})]

In [123]:
# convert the track_info dictionary to a dataframe where the keys become 'track_id' column
collected_track_info_df = pd.DataFrame.from_dict(track_info, orient='index')
# rename the columns
collected_track_info_df.reset_index(inplace=True)
collected_track_info_df.rename(columns={'index':'track_id'}, inplace=True)

In [124]:
# add the collected track info to the track_info_df (if it exists)
if tracks_info_df is not None:
    tracks_info_df = pd.concat([tracks_info_df, collected_track_info_df], ignore_index=True)
else:
    tracks_info_df = pd.DataFrame.from_dict(track_info, orient='index')

In [125]:
len(tracks_info_df)

5723

### Convert the dict to dataframe and save it to csv

In [126]:
# save the tracks_info_df to csv
tracks_info_df.to_csv('datasets/tracks_info_df.csv', index=False)

## Collecting Track Audio Features

April 2025 update: audio features api is deprecated, I'll update this section later

In [134]:
track_features = {}
for i in range(0, len(tracks_uri_list), 50):
    track_batch = tracks_uri_list[i:i+50]
    sapi = sp.audio_features(track_batch)
    for track in sapi:
        if track is not None:
            track_features[track['id']] = {'danceability':track['danceability'], 'energy':track['energy'], 'key':track['key'], 'loudness':track['loudness'], 'mode':track['mode'], 'speechiness':track['speechiness'], 'acousticness':track['acousticness'], 'instrumentalness':track['instrumentalness'], 'liveness':track['liveness'], 'valence':track['valence'], 'tempo':track['tempo'], 'time_signature':track['time_signature']}
        # we'll skip it if the track has no audio features data
    time.sleep(1)

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=0ftCbviWA5aWopBjsBK6Pv,6dOtVTDdiauQNBQEDOtlAB,6cBbx14y271qwBX2CQttwQ,1OYA4DbZ6IOtSVuKb0VFVQ,4IqrPu63viykyz34gUYKAx,5VBjyOQzqlPNgdRPMM6prF,5X9ZLed3uTNmlv5CxSc1XK,5Rx6U54oaGNEZ8eVy3VLK7,2RE8lrdEopwUuCfphJKnf8,1KMEDSIl2j1NwYa9mgvMyg,7hhRZJkO3jBxwxzGgurZ2Q,4sblRQJa9p5B70Wt1ZIcCq,1l1z45twP4mVGQL9UNILoq,6q8nlNnBLSAn5tU6tH9Zlz,2OzhQlSqBEmt7hmkYxfT6m,1207xP3DSoGjDRr7Nc0ohN,2hFcaybCYlGLn9elIsTuc3,4mL3gs1HONGGLaZyW6OYMq,4bx0ZIQecKaet7Jtnes6KT,6eOc29YX6IoRO2gwPoUjAs,0gggjVyUJd90tYzAnmBjFV,7LeYstsBY4QQa4rdseQA1U,0s39oFQ4PLIiecJaI0qRfV,5GpEHUNI0T7L7H3DnAaBXh,6jEi6mO39kO1WtOm6ksogj,5XpbhQtOI92vbhjrRtDOjF,09cWMixCAoFoFMmegG4wqf,42Ly3cfDuDaLZGvgOTwvLi,6ibSWqU0gfg8J5qxjepe9W,6oPDWUwxTIsGejcHd9tLdc,39fD0qvjgk8RarJnoBiDTx,6Ac4h5tnkLkHHlFyBfL3j4,5muDXHVGjNPaPYJlHZClG7,5I8wqCZfqMXhJ8NYwFHgwc,2ZvnX3tcCm4qwcvUuvZBvq,31kXmImsj99zlKKjqkaVlW,7JgDvGmgeKcdM633G1XAo9,2ttemPsvqBdA8ItME6DCbL,63reuc8nVqfO3bmxCLUKDq,1cIi1x8ANjMO97wH5OfYO1,059gVW493dU

SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=0ftCbviWA5aWopBjsBK6Pv,6dOtVTDdiauQNBQEDOtlAB,6cBbx14y271qwBX2CQttwQ,1OYA4DbZ6IOtSVuKb0VFVQ,4IqrPu63viykyz34gUYKAx,5VBjyOQzqlPNgdRPMM6prF,5X9ZLed3uTNmlv5CxSc1XK,5Rx6U54oaGNEZ8eVy3VLK7,2RE8lrdEopwUuCfphJKnf8,1KMEDSIl2j1NwYa9mgvMyg,7hhRZJkO3jBxwxzGgurZ2Q,4sblRQJa9p5B70Wt1ZIcCq,1l1z45twP4mVGQL9UNILoq,6q8nlNnBLSAn5tU6tH9Zlz,2OzhQlSqBEmt7hmkYxfT6m,1207xP3DSoGjDRr7Nc0ohN,2hFcaybCYlGLn9elIsTuc3,4mL3gs1HONGGLaZyW6OYMq,4bx0ZIQecKaet7Jtnes6KT,6eOc29YX6IoRO2gwPoUjAs,0gggjVyUJd90tYzAnmBjFV,7LeYstsBY4QQa4rdseQA1U,0s39oFQ4PLIiecJaI0qRfV,5GpEHUNI0T7L7H3DnAaBXh,6jEi6mO39kO1WtOm6ksogj,5XpbhQtOI92vbhjrRtDOjF,09cWMixCAoFoFMmegG4wqf,42Ly3cfDuDaLZGvgOTwvLi,6ibSWqU0gfg8J5qxjepe9W,6oPDWUwxTIsGejcHd9tLdc,39fD0qvjgk8RarJnoBiDTx,6Ac4h5tnkLkHHlFyBfL3j4,5muDXHVGjNPaPYJlHZClG7,5I8wqCZfqMXhJ8NYwFHgwc,2ZvnX3tcCm4qwcvUuvZBvq,31kXmImsj99zlKKjqkaVlW,7JgDvGmgeKcdM633G1XAo9,2ttemPsvqBdA8ItME6DCbL,63reuc8nVqfO3bmxCLUKDq,1cIi1x8AN

In [48]:
# get first 5 items in the dictionary
list(track_features.items())[:5]

[('6tAM5c0bJOwRqGAEgiNMpI',
  {'danceability': 0.446,
   'energy': 0.823,
   'key': 11,
   'loudness': -5.279,
   'mode': 0,
   'speechiness': 0.0859,
   'acousticness': 0.00256,
   'instrumentalness': 0.0748,
   'liveness': 0.117,
   'valence': 0.263,
   'tempo': 128.198,
   'time_signature': 4}),
 ('5Q0P0cX3e42PgKd8LLS3ms',
  {'danceability': 0.402,
   'energy': 0.856,
   'key': 1,
   'loudness': -4.256,
   'mode': 1,
   'speechiness': 0.0659,
   'acousticness': 0.00504,
   'instrumentalness': 0.0164,
   'liveness': 0.253,
   'valence': 0.247,
   'tempo': 140.028,
   'time_signature': 4}),
 ('4VrdksXJVhAOLW49qV0VTQ',
  {'danceability': 0.426,
   'energy': 0.915,
   'key': 11,
   'loudness': -3.881,
   'mode': 0,
   'speechiness': 0.143,
   'acousticness': 0.000899,
   'instrumentalness': 0.108,
   'liveness': 0.495,
   'valence': 0.271,
   'tempo': 130.405,
   'time_signature': 4}),
 ('0ng42pTjKgskmobNzhnEUa',
  {'danceability': 0.434,
   'energy': 0.839,
   'key': 8,
   'loudness': 

### Convert the dict to dataframe and save it to csv

In [49]:
track_features_df = pd.DataFrame.from_dict(track_features, orient='index')
track_features_df.to_csv('datasets/tracks_features_df.csv')

## Collecting Track Lyrics (using LyricsGenius / Genius API)

here, we are going to use [Genius API](https://genius.com/developers) through [LyricsGenius](https://github.com/johnwmillr/LyricsGenius) to get the lyrics of the tracks, you can get the API key by registering at the website.

In [4]:
import lyricsgenius
import re
import time

In [5]:
lyricsgenius_client = lyricsgenius.Genius(GENIUS_ACCESS_TOKEN)

In [6]:
tracks_info_df = pd.read_csv('datasets/tracks_info_df.csv')
print(tracks_info_df.shape)

(5245, 6)


In [7]:
tracks_info_df = tracks_info_df[~tracks_info_df.isna().any(axis=1)].drop_duplicates(subset='track_id')
print(tracks_info_df.shape)

(5243, 6)


In [8]:
tracks_info_df.head(5)

,track_id,name,artistName,release_date,popularity,duration_ms
0,6tAM5c0bJOwRqGAEgiNMpI,Solace Album Mix,Monstercat,2012-06-06,0,3538579
1,4mjgNE8R31AzxWfPNGtVMf,Best of 2015 (Album Mix),Monstercat,2016-01-22,0,9158194
2,5Q0P0cX3e42PgKd8LLS3ms,Horizon Album Mix,Monstercat,2014-08-06,0,3623121
3,6jvMmRtSzoEibQGrQkSISQ,Monstercat Best of 2012,Monstercat,2013-02-04,0,6348017
4,1KzLyjpjIRHuuj4iX8QsC2,Monstercat Podcast EP. 100,Monstercat,2016-04-05,0,9744610


In [16]:
lyrics_data = {}
rate_limit_reached = False

# Loop through the list of unique isrc values
for i in range(0, len(tracks_info_df)):
    while True:
        try:
            song = lyricsgenius_client.search_song(tracks_info_df.iloc[i]['name'], tracks_info_df.iloc[i]['artistName'])
            track_id = tracks_info_df.iloc[i]['track_id']

            if song is not None:
                lyrics_data[track_id] = {
                    'title_genius': song.title,
                    'artist_genius': song.artist,
                    'title_spotify': tracks_info_df.iloc[i]['name'],
                    'artist_spotify': tracks_info_df.iloc[i]['artistName'],

                    # remove garbage character from ads in the lyrics
                    'lyrics': re.sub(r"(?<!\n)\n(\[)", r"\n\n\1", song.lyrics)
                }
            else:
                lyrics_data[track_id] = {
                    'title_genius': None,
                    'artist_genius': None,
                    'title_spotify': tracks_info_df.iloc[i]['name'],
                    'artist_spotify': tracks_info_df.iloc[i]['artistName'],
                    'lyrics': None
                }

            rate_limit_reached = False
            break  # Exit the retry loop if the request is successful

        except Exception as e:
            print(f"An error occurred: {e}")
            if not rate_limit_reached:
                rate_limit_reached = True
            time.sleep(60)  # Wait for 60 seconds before retrying

# Save the data to a CSV file
lyrics_data_df = pd.DataFrame.from_dict(lyrics_data, orient='index')
lyrics_data_df.to_csv('datasets/lyrics_data.csv')

Searching for "きっとまたいつか" by DEPAPEPE...
No results found for: 'きっとまたいつか DEPAPEPE'
Searching for "晴れ 時どき 雪" by DEPAPEPE...
No results found for: '晴れ 時どき 雪 DEPAPEPE'
Searching for "SPARK!" by DEPAPEPE...
Done.
Searching for "Dragon Night" by SEKAI NO OWARI...
Done.
Searching for "生物学的幻想曲" by SEKAI NO OWARI...
Done.
Searching for "眠り姫" by SEKAI NO OWARI...
Done.
Searching for "Moonlight Station - Remixed by Dux Content from London" by SEKAI NO OWARI...
No results found for: 'Moonlight Station - Remixed by Dux Content from London SEKAI NO OWARI'
Searching for "RAIN" by SEKAI NO OWARI...
Done.
Searching for "Introducing the College" by Takatsugu Muramatsu...
No results found for: 'Introducing the College Takatsugu Muramatsu'
Searching for "RPG" by SEKAI NO OWARI...
Done.
Searching for "笑顔" by Ikimonogakari...
Done.
Searching for "キミがいる" by Ikimonogakari...
Done.
Searching for "ブルーバード" by Ikimonogakari...
Done.
Searching for "ハジマリノウタ〜遠い空澄んで〜" by Ikimonogakari...
Done.
Searching for "SAKURA" 